# 05 - Demand Forecasting: Machine Learning & Feature Engineering

Notebook ini bertujuan untuk membangun model **Demand Forecasting** berbasis Machine Learning dari data agregasi harian (`daily_demand`). 

**Tahapan Utama:**
1. Mengambil data transaksi harian langsung dari PostgreSQL.
2. Melakukan penggabungan (*enrichment*) dengan master produk, toko, kalender, dan cuaca tanpa duplikasi data.
3. Melakukan *Feature Engineering* (fitur waktu, lag demand, dan rolling statistics dengan pencegahan *data leakage*).
4. Menyiapkan dataset ML dengan membuang baris bernilai *null* akibat perhitungan lag.
5. Melakukan pemisahan data latih & uji (*Train-Test Split*) berbasis waktu/temporal.
6. Membangun model **Baseline (Lag-7)** sebagai pembanding.
7. Membangun dan mengevaluasi model **Random Forest Regressor**.
8. Menganalisis **Feature Importance** untuk interpretasi model.

## 1. Setup Environment & Load Daily Demand Data

Menginisialisasi koneksi database PostgreSQL dan mengambil data transaksi harian teragregasi per tanggal, toko, dan produk.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Set path root project untuk mengimpor modul custom
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from etl.db_connection import get_engine

# Inisialisasi koneksi PostgreSQL
engine = get_engine()

# Query SQL agregasi demand harian per tanggal, toko, dan produk
query = """
SELECT
    s.transaction_date,
    s.store_id,
    s.product_id,
    SUM(s.transaction_qty) AS demand,
    SUM(s.transaction_qty * s.unit_price) AS revenue,
    COUNT(DISTINCT s.transaction_id) AS transactions
FROM sales_transactions s
GROUP BY
    s.transaction_date,
    s.store_id,
    s.product_id
ORDER BY
    s.transaction_date,
    s.store_id,
    s.product_id;
"""

# Read data dari database
daily = pd.read_sql(query, engine, parse_dates=["transaction_date"])

print("Rows:", len(daily))

Rows: 31764


## 2. Enrichment: Merge Product, Store, Calendar, & Weather

Menggabungkan atribut produk, lokasi toko, kalender, dan cuaca ke dalam dataset harian. Penggabungan menggunakan validasi `many_to_one` untuk menjamin tidak ada lonjakan/duplikasi baris.

In [2]:
# Ambil tabel dimensi dari PostgreSQL
products = pd.read_sql(
    """
    SELECT
        p.product_id,
        p.product_type,
        p.product_detail,
        c.category_name
    FROM products p
    JOIN categories c
        ON p.category_id = c.category_id;
    """,
    engine,
)

stores = pd.read_sql(
    """
    SELECT
        store_id,
        store_location
    FROM stores;
    """,
    engine,
)

calendar = pd.read_sql(
    """
    SELECT *
    FROM calendar;
    """,
    engine,
    parse_dates=["date"],
)

weather = pd.read_sql(
    """
    SELECT *
    FROM weather;
    """,
    engine,
    parse_dates=["date"],
)

# Merge dengan pengujian integritas data (validate="many_to_one")
daily = daily.merge(
    products, on="product_id", how="left", validate="many_to_one"
)
daily = daily.merge(stores, on="store_id", how="left", validate="many_to_one")
daily = daily.merge(
    calendar,
    left_on="transaction_date",
    right_on="date",
    how="left",
    validate="many_to_one",
)
daily = daily.merge(
    weather,
    left_on=["transaction_date", "store_location"],
    right_on=["date", "location"],
    how="left",
    validate="many_to_one",
)

# Validasi akhir enrichment
print("Rows after enrichment:", len(daily))
print("Missing values:", daily.isna().sum().sum())

Rows after enrichment: 31764
Missing values: 0


## 3. Feature Engineering: Time Features

Ekstraksi komponen waktu dari `transaction_date` untuk menangkap tren musiman harian, mingguan, dan siklus bulanan.

In [3]:
# Ekstraksi fitur komponen tanggal
daily["day_of_month"] = daily["transaction_date"].dt.day
daily["week_of_year"] = (
    daily["transaction_date"].dt.isocalendar().week.astype(int)
)
daily["is_month_start"] = (
    daily["transaction_date"].dt.is_month_start.astype(int)
)
daily["is_month_end"] = daily["transaction_date"].dt.is_month_end.astype(int)

# Tampilkan beberapa kolom tanggal baru
daily[["transaction_date", "day_of_month", "week_of_year", "is_month_start", "is_month_end"]].head()

,transaction_date,day_of_month,week_of_year,is_month_start,is_month_end
0,2023-01-01,1,52,1,0
1,2023-01-01,1,52,1,0
2,2023-01-01,1,52,1,0
3,2023-01-01,1,52,1,0
4,2023-01-01,1,52,1,0


## 4. Feature Engineering: Historical Demand (Lag & Rolling)

Membuat fitur historis per kombinasi `store_id x product_id`. 
* **Catatan Penting:** Penggunaan `shift(1)` pada fitur rolling sangat krusial untuk menghindari **Data Leakage** (mencegah model melihat demand hari ini saat melakukan prediksi harian).

In [4]:
# Urutkan data berdasarkan store, product, dan tanggal transaksi
daily = daily.sort_values(["store_id", "product_id", "transaction_date"])

# Buat grup agregasi per kombinasi store dan product
group = daily.groupby(["store_id", "product_id"])["demand"]

# Fitur Lag (demand H-1, H-7, H-14, H-28)
daily["lag_1"] = group.shift(1)
daily["lag_7"] = group.shift(7)
daily["lag_14"] = group.shift(14)
daily["lag_28"] = group.shift(28)

# Fitur Rolling Mean dengan shift(1) untuk mencegah data leakage
daily["rolling_mean_7"] = (
    daily.groupby(["store_id", "product_id"])["demand"].transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)
daily["rolling_mean_28"] = (
    daily.groupby(["store_id", "product_id"])["demand"].transform(
        lambda x: x.shift(1).rolling(28).mean()
    )
)

# Tampilkan sampel data fitur lag dan rolling
daily[["store_id", "product_id", "transaction_date", "demand", "lag_1", "lag_7", "rolling_mean_7"]].tail()

,store_id,product_id,transaction_date,demand,lag_1,lag_7,rolling_mean_7
31090,8,87,2023-06-26,18,19.0,16.0,18.000000
31261,8,87,2023-06-27,24,18.0,6.0,18.285714
31416,8,87,2023-06-28,11,24.0,25.0,20.857143
31569,8,87,2023-06-29,19,11.0,17.0,18.857143
31763,8,87,2023-06-30,12,19.0,19.0,19.142857


## 5. Machine Learning Dataset Formulation

Menyeleksi fitur-fitur final yang akan digunakan oleh model ML dan menghapus baris awal yang mengandung *missing value* akibat keterbatasan histori lag (di bawah 28 hari pertama).

In [5]:
# Definisikan daftar fitur dan target
feature_columns = [
    "store_id",
    "product_id",
    "category_name",
    "temperature",
    "rainfall_mm",
    "is_weekend",
    "month",
    "day_of_week",
    "week_of_year",
    "is_month_start",
    "is_month_end",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
]
target = "demand"

# Salin subset kolom yang dibutuhkan ke DataFrame ml_data
ml_data = daily[feature_columns + ["transaction_date", target]].copy()

# Buang baris bernilai null yang timbul dari perhitungan lag
ml_data = ml_data.dropna(
    subset=[
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
    ]
)

# Cek jumlah baris dan jumlah fitur
print("ML rows:", len(ml_data))
print("Features:", len(feature_columns))

ML rows: 25113
Features: 17


## 6. Time-based Train-Test Split

Memisah dataset menjadi data **Train** (Januari - April) dan **Test** (Mei - Juni) berdasarkan cutoff tanggal `2023-05-01`. Pembagian berbasis waktu ini wajib dilakukan dalam proyek *Time Series Forecasting* agar simulasi evaluasi mencerminkan kondisi riil di masa depan.

In [6]:
# Tentukan cutoff tanggal pemisahan
cutoff_date = pd.Timestamp("2023-05-01")

# Splitting berbasis waktu
train = ml_data[ml_data["transaction_date"] < cutoff_date].copy()
test = ml_data[ml_data["transaction_date"] >= cutoff_date].copy()

print("Train:", len(train))
print("Test :", len(test))

Train: 14292
Test : 10821


## 7. Baseline Model (Naive Lag-7)

Mengevaluasi performa model sederhana (*naive baseline*) dengan mengasumsikan bahwa demand hari ini persis sama dengan demand 7 hari yang lalu (`lag_7`). Angka ini menjadi *benchmark* minimal yang harus dikalahkan oleh model Machine Learning.

In [7]:
# Prediksi baseline menggunakan nilai lag_7
baseline_predictions = test["lag_7"]

# Hitung metrik evaluasi MAE dan RMSE
mae_base = mean_absolute_error(test["demand"], baseline_predictions)
rmse_base = np.sqrt(mean_squared_error(test["demand"], baseline_predictions))

print("Baseline MAE :", mae_base)
print("Baseline RMSE:", rmse_base)

Baseline MAE : 4.430736530819702
Baseline RMSE: 5.958137235423881


## 8. Random Forest Regressor Model & Performance Comparison

Melatih model **Random Forest Regressor** dengan fitur kategorikal yang telah di-encode (`One-Hot Encoding`), kemudian membandingkan performanya secara langsung dengan model Baseline.

In [8]:
# One-Hot Encoding untuk kolom kategorikal 'category_name'
categorical_columns = ["category_name"]

X_train = pd.get_dummies(
    train[feature_columns], columns=categorical_columns
)
X_test = pd.get_dummies(test[feature_columns], columns=categorical_columns)

# Samakan struktur kolom X_test agar presisi dengan X_train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train[target]
y_test = test[target]

# Inisialisasi dan fitting model Random Forest
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

# Prediksi data uji
predictions = model.predict(X_test)

# Evaluasi model Random Forest
mae_rf = mean_absolute_error(y_test, predictions)
rmse_rf = np.sqrt(mean_squared_error(y_test, predictions))

# Tabel Perbandingan Performa Model
comparison = pd.DataFrame(
    {
        "model": ["Baseline Lag-7", "Random Forest"],
        "MAE": [mae_base, mae_rf],
        "RMSE": [rmse_base, rmse_rf],
    }
)
comparison

,model,MAE,RMSE
0,Baseline Lag-7,4.430737,5.958137
1,Random Forest,3.336788,4.534847


## 9. Feature Importance Analysis

Menganalisis tingkat kepentingikatan (*importance*) dari masing-masing fitur dalam model Random Forest untuk mengonfirmasi faktor penentu utama permintaan barang.

In [9]:
# Ekstraksi dan pengurutan nilai Feature Importance
feature_importance = pd.DataFrame(
    {"feature": X_train.columns, "importance": model.feature_importances_}
).sort_values("importance", ascending=False)

# Tampilkan 15 fitur paling berpengaruh
feature_importance.head(15)

,feature,importance
15,rolling_mean_28,0.310354
5,month,0.079076
14,rolling_mean_7,0.075899
1,product_id,0.075332
2,temperature,0.064315
13,lag_28,0.063578
11,lag_7,0.059296
10,lag_1,0.058765
12,lag_14,0.054483
3,rainfall_mm,0.051285
